<a href="https://colab.research.google.com/github/jamie07262/INFO-3608-PROJECT/blob/main/New_Data_Processor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import requests
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
# GitHub API URL for the directory
api_url = "https://api.github.com/repos/jamie07262/INFO-3608-PROJECT/contents/data/raw"

# Create local directory
os.makedirs("raw", exist_ok=True)

# Get list of files from GitHub API
response = requests.get(api_url)
if response.status_code == 200:
    files = response.json()
    for file in files:
        if file['name'].endswith('.csv'):
            download_url = file['download_url']
            r = requests.get(download_url)
            with open(f"raw/{file['name']}", "wb") as f:
                f.write(r.content)
            print(f"Downloaded: {file['name']}")
else:
    print(f"Failed to access API: {response.status_code}")

print("All CSV files downloaded.")

Downloaded: 538ratingsMen.csv
Downloaded: 538ratingsWomen.csv
Downloaded: Cities.csv
Downloaded: Conferences.csv
Downloaded: MConferenceTourneyGames.csv
Downloaded: MGameCities.csv
Downloaded: MMasseyOrdinalsPart1.csv
Downloaded: MMasseyOrdinalsPart2.csv
Downloaded: MNCAATourneyCompactResults.csv
Downloaded: MNCAATourneyDetailedResults.csv
Downloaded: MNCAATourneySeedRoundSlots.csv
Downloaded: MNCAATourneySeeds.csv
Downloaded: MNCAATourneySlots.csv
Downloaded: MRegularSeasonCompactResults.csv
Downloaded: MRegularSeasonDetailedResults.csv
Downloaded: MSeasons.csv
Downloaded: MSecondaryTourneyCompactResults.csv
Downloaded: MSecondaryTourneyTeams.csv
Downloaded: MTeamCoaches.csv
Downloaded: MTeamConferences.csv
Downloaded: MTeamSpellings.csv
Downloaded: MTeams.csv
Downloaded: SampleSubmissionStage1.csv
Downloaded: SampleSubmissionStage2.csv
Downloaded: SeedBenchmarkStage1.csv
Downloaded: WConferenceTourneyGames.csv
Downloaded: WGameCities.csv
Downloaded: WNCAATourneyCompactResults.csv
Dow

In [10]:
#Read and Merge data

seasonResults = pd.concat(
    [
        pd.read_csv("raw/MRegularSeasonDetailedResults.csv").assign(Divison = 1),
        pd.read_csv("raw/WRegularSeasonDetailedResults.csv").assign(Divison = 0),
    ]
).reset_index(drop=True)

tourneyResults = pd.concat(
    [
        pd.read_csv("raw/MNCAATourneyDetailedResults.csv").assign(Divison = 1),
        pd.read_csv("raw/WNCAATourneyDetailedResults.csv").assign(Divison = 0),
    ]
).reset_index(drop=True)

seeds = pd.concat(
    [
        pd.read_csv("raw/MNCAATourneySeeds.csv").assign(Divison = 1),
        pd.read_csv("raw/WNCAATourneySeeds.csv").assign(Divison = 0),
    ]
).reset_index(drop=True)

In [19]:
pd.options.display.max_columns = None

In [ ]:
seasonResults

In [ ]:
tourneyResults

In [ ]:
seeds

In [11]:
#Remove seeds before 2003 to match season and tourney
seeds = seeds.loc[seeds["Season"] >= 2003]

In [ ]:
seeds

# Data Explination:

Team Box Scores are provided in "Detailed Results" files rather than "Compact Results" files. However, the two files are strongly related.
In a Detailed Results file, the first eight columns (Season, DayNum, WTeamID, WScore, LTeamID, LScore, WLoc, and NumOT) are exactly the same as a Compact Results file. However, in a Detailed Results file, there are many additional columns. The column names should be self-explanatory to basketball fans (as above, "W" or "L" refers to the winning or losing team):

    WFGM - field goals made (by the winning team)
    WFGA - field goals attempted (by the winning team)
    WFGM3 - three pointers made (by the winning team)
    WFGA3 - three pointers attempted (by the winning team)
    WFTM - free throws made (by the winning team)
    WFTA - free throws attempted (by the winning team)
    WOR - offensive rebounds (pulled by the winning team)
    WDR - defensive rebounds (pulled by the winning team)
    WAst - assists (by the winning team)
    WTO - turnovers committed (by the winning team)
    WStl - steals (accomplished by the winning team)
    WBlk - blocks (accomplished by the winning team)
    WPF - personal fouls committed (by the winning team)

(and then the same set of stats from the perspective of the losing team: LFGM is the number of field goals made by the losing team, and so on up to LPF).

In [22]:
#Swap team positions in box scores
def prepareData(dataFrame):
    dataFrame = dataFrame[["Season", "DayNum", "LTeamID", "LScore", "WTeamID", "WScore", "NumOT",
            "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF",
            "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF", "Divison"]].copy()


    #Adjustments for Overtime (Since more time = more stats)
    overtimeAdjustment = (40 + 5 * dataFrame["NumOT"]) / 40
    columnAdjustments = ["LScore", "WScore",
               "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF",
               "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF", "Divison"]
    for col in columnAdjustments:
        dataFrame[col] = dataFrame[col] / overtimeAdjustment

    dataFrameSwap = dataFrame.copy()
    final = pd.concat([dataFrame, dataFrameSwap]).reset_index(drop=True)
    final["PointDifference"] = final["WScore"] - final["LScore"]
    final["Win"] = (final["PointDifference"] > 0) * 1

    final = final[["Season", "DayNum", "LTeamID", "LScore", "WTeamID", "WScore", "NumOT",
            "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF",
            "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF", "PointDifference", "Win", "Divison"]].copy()
    return final

seasonData = prepareData(seasonResults)
tourneyData = prepareData(tourneyResults)

In [23]:
seasonData

,Season,DayNum,LTeamID,LScore,WTeamID,WScore,NumOT,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,PointDifference,Win,Divison
0,2003,10,1328,62.0,1104,68.0,0,22.0,53.0,2.0,10.0,16.0,22.0,10.0,22.0,8.0,18.0,9.0,2.0,20.0,27.0,58.0,3.0,14.0,11.0,18.0,14.0,24.0,13.0,23.0,7.0,1.0,22.0,6.0,1,1.0
1,2003,10,1393,63.0,1272,70.0,0,24.0,67.0,6.0,24.0,9.0,20.0,20.0,25.0,7.0,12.0,8.0,6.0,16.0,26.0,62.0,8.0,20.0,10.0,19.0,15.0,28.0,16.0,13.0,4.0,4.0,18.0,7.0,1,1.0
2,2003,11,1437,61.0,1266,73.0,0,22.0,73.0,3.0,26.0,14.0,23.0,31.0,22.0,9.0,12.0,2.0,5.0,23.0,24.0,58.0,8.0,18.0,17.0,29.0,17.0,26.0,15.0,10.0,5.0,2.0,25.0,12.0,1,1.0
3,2003,11,1457,50.0,1296,56.0,0,18.0,49.0,6.0,22.0,8.0,15.0,17.0,20.0,9.0,19.0,4.0,3.0,23.0,18.0,38.0,3.0,9.0,17.0,31.0,6.0,19.0,11.0,12.0,14.0,2.0,18.0,6.0,1,1.0
4,2003,11,1208,71.0,1400,77.0,0,24.0,62.0,6.0,16.0,17.0,27.0,21.0,15.0,12.0,10.0,7.0,1.0,14.0,30.0,61.0,6.0,14.0,11.0,13.0,17.0,22.0,12.0,14.0,4.0,4.0,20.0,6.0,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401175,2025,131,3413,66.0,3471,75.0,0,24.0,67.0,9.0,29.0,9.0,14.0,9.0,26.0,14.0,10.0,6.0,5.0,22.0,26.0,62.0,4.0,19.0,19.0,28.0,8.0,31.0,10.0,11.0,6.0,1.0,20.0,9.0,1,0.0
401176,2025,132,3476,49.0,3192,66.0,0,21.0,57.0,4.0,20.0,3.0,4.0,14.0,22.0,14.0,17.0,4.0,1.0,17.0,23.0,55.0,3.0,21.0,17.0,18.0,10.0,22.0,11.0,9.0,8.0,1.0,8.0,17.0,1,0.0
401177,2025,132,3119,62.0,3250,74.0,0,25.0,56.0,6.0,17.0,6.0,10.0,8.0,13.0,10.0,10.0,5.0,0.0,20.0,27.0,45.0,5.0,14.0,15.0,17.0,5.0,25.0,15.0,15.0,6.0,0.0,12.0,12.0,1,0.0
401178,2025,132,3125,62.0,3293,83.0,0,24.0,68.0,2.0,21.0,12.0,14.0,12.0,22.0,11.0,7.0,5.0,0.0,16.0,28.0,54.0,14.0,28.0,13.0,15.0,5.0,33.0,21.0,13.0,2.0,3.0,15.0,21.0,1,0.0


In [24]:
tourneyData

,Season,DayNum,LTeamID,LScore,WTeamID,WScore,NumOT,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,PointDifference,Win,Divison
0,2003,134,1411,74.666667,1421,81.777778,1,25.777778,59.555556,10.666667,27.555556,12.444444,27.555556,15.111111,24.888889,14.222222,13.333333,4.444444,0.000000,19.555556,28.444444,61.333333,9.777778,25.777778,15.111111,23.111111,12.444444,26.666667,15.111111,10.666667,4.444444,2.666667,19.555556,7.111111,1,0.888889
1,2003,136,1436,51.000000,1112,80.000000,0,20.000000,64.000000,4.000000,16.000000,7.000000,7.000000,8.000000,26.000000,12.000000,17.000000,10.000000,3.000000,15.000000,31.000000,66.000000,7.000000,23.000000,11.000000,14.000000,11.000000,36.000000,22.000000,16.000000,10.000000,7.000000,8.000000,29.000000,1,1.000000
2,2003,136,1272,71.000000,1113,84.000000,0,25.000000,69.000000,7.000000,28.000000,14.000000,21.000000,20.000000,22.000000,11.000000,12.000000,2.000000,5.000000,18.000000,31.000000,59.000000,6.000000,14.000000,16.000000,22.000000,10.000000,27.000000,18.000000,9.000000,7.000000,4.000000,19.000000,13.000000,1,1.000000
3,2003,136,1166,73.000000,1141,79.000000,0,27.000000,60.000000,7.000000,17.000000,12.000000,17.000000,14.000000,17.000000,20.000000,21.000000,6.000000,6.000000,21.000000,29.000000,53.000000,3.000000,7.000000,18.000000,25.000000,11.000000,20.000000,15.000000,18.000000,13.000000,1.000000,19.000000,6.000000,1,1.000000
4,2003,136,1301,65.777778,1143,67.555556,1,22.222222,49.777778,8.000000,18.666667,13.333333,17.777778,8.888889,23.111111,14.222222,12.444444,4.444444,7.111111,16.888889,24.000000,56.888889,6.222222,17.777778,13.333333,20.444444,16.000000,17.777778,15.111111,11.555556,7.111111,1.777778,12.444444,1.777778,1,0.888889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4547,2024,147,3425,73.000000,3163,80.000000,0,23.000000,70.000000,9.000000,29.000000,18.000000,20.000000,10.000000,25.000000,10.000000,9.000000,6.000000,4.000000,20.000000,28.000000,58.000000,7.000000,15.000000,17.000000,27.000000,5.000000,30.000000,17.000000,12.000000,6.000000,5.000000,21.000000,7.000000,1,0.000000
4548,2024,147,3261,87.000000,3234,94.000000,0,34.000000,88.000000,8.000000,24.000000,11.000000,17.000000,21.000000,28.000000,15.000000,13.000000,6.000000,6.000000,21.000000,32.000000,69.000000,13.000000,31.000000,17.000000,22.000000,3.000000,29.000000,16.000000,11.000000,6.000000,3.000000,15.000000,7.000000,1,0.000000
4549,2024,151,3163,69.000000,3234,71.000000,0,29.000000,63.000000,8.000000,25.000000,3.000000,4.000000,6.000000,22.000000,21.000000,14.000000,15.000000,1.000000,18.000000,27.000000,59.000000,7.000000,25.000000,10.000000,14.000000,9.000000,23.000000,12.000000,16.000000,7.000000,1.000000,9.000000,2.000000,1,0.000000
4550,2024,151,3301,59.000000,3376,78.000000,0,20.000000,62.000000,6.000000,23.000000,13.000000,18.000000,10.000000,18.000000,5.000000,12.000000,9.000000,1.000000,9.000000,33.000000,66.000000,8.000000,19.000000,4.000000,4.000000,10.000000,34.000000,18.000000,15.000000,10.000000,6.000000,16.000000,19.000000,1,0.000000
